# visualizing-disruptive-forces

**Author:** Sana Ur Rehman  
**Profession:** Data Scientist  
**Created:** 2026  

---

## License

This project is licensed under the **MIT License**. 

You are free to use, modify, distribute, and build upon this work for both commercial and non-commercial purposes, provided you give appropriate **credit** to the original author. For the full legal text and conditions, please refer to the `LICENSE` file included in this project's repository.

---

## Citation

If you reference or build upon this project, please provide appropriate credit.

For formal citation information, please see the project's `README.md` and `CITATION.cff` files.

# Data Preprocessing and Source Selection

## Purpose

This notebook prepares the Bank for International Settlements (BIS) Consolidated Banking Statistics for the *Visualizing Disruptive Forces* group project.

The project analyzes the cross-border banking network during the 2008–09 financial crisis. Countries are treated as network nodes, and cross-border lending relationships are treated as weighted and directed links. The project requires empirical claims to be supported by values obtained from the Global Banking Network Risk Dashboard and the underlying BIS Consolidated Banking Statistics.

Before building the network, this notebook compares three downloaded files:

1. `consolidated banking statistics.csv` — the BIS bulk download containing multiple reporting countries, counterparty countries, measures, reporting bases, and quarterly observations.
2. `2007-Q1.csv` — a dashboard export from BIS Table B4 for the first quarter of 2007.
3. `2008-Q1.csv` — a dashboard export from BIS Table B4 for the first quarter of 2008.

The purpose of this comparison is to determine which file is suitable as the main analytical data source and how the dashboard exports can be used for validation and documentation.

In [ ]:
import os
print(os.getcwd())

define path

In [2]:
from pathlib import Path
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 100)

RAW_DIR = Path("../data/raw")

bulk_path = RAW_DIR / "consolidated banking statistics.csv"
q1_2007_path = RAW_DIR / "2007-Q1.csv"
q1_2008_path = RAW_DIR / "2008-Q1.csv"

for path in [bulk_path, q1_2007_path, q1_2008_path]:
    print(path, "exists:", path.exists())

..\data\raw\consolidated banking statistics.csv exists: True
..\data\raw\2007-Q1.csv exists: True
..\data\raw\2008-Q1.csv exists: True


read the csv files

In [5]:
q1_2007 = pd.read_csv(q1_2007_path, low_memory=False)
q1_2008 = pd.read_csv(q1_2008_path, low_memory=False)

bulk = pd.read_csv(bulk_path, low_memory=False)

print("2007-Q1 shape:", q1_2007.shape)
print("2008-Q1 shape:", q1_2008.shape)
print("Bulk shape:", bulk.shape)

2007-Q1 shape: (75, 23)
2008-Q1 shape: (75, 23)
Bulk shape: (228370, 167)


inspect column names

In [6]:
def show_columns(df, name):
    print(f"\n{name}")
    print("=" * len(name))
    for i, col in enumerate(df.columns):
        print(f"{i}: {col!r}")

show_columns(q1_2007, "2007-Q1")
show_columns(q1_2008, "2008-Q1")
show_columns(bulk, "Bulk dataset")


2007-Q1
0: 'Publication Table'
1: 'Publication table description'
2: 'Period'
3: 'Dimension'
4: 'Dimension name'
5: 'Dimension code'
6: 'Dimension value'
7: 'Dimension.1'
8: 'Dimension name.1'
9: 'Dimension code.1'
10: 'Dimension value.1'
11: 'Dimension.2'
12: 'Dimension name.2'
13: 'Dimension code.2'
14: 'Dimension value.2'
15: 'Dataflow'
16: 'Dataflow name'
17: 'Last update of data'
18: 'Downloaded at'
19: 'Source'
20: 'Source URL'
21: 'Download URL'
22: 'About International banking / Consolidated banking statistics'

2008-Q1
0: 'Publication Table'
1: 'Publication table description'
2: 'Period'
3: 'Dimension'
4: 'Dimension name'
5: 'Dimension code'
6: 'Dimension value'
7: 'Dimension.1'
8: 'Dimension name.1'
9: 'Dimension code.1'
10: 'Dimension value.1'
11: 'Dimension.2'
12: 'Dimension name.2'
13: 'Dimension code.2'
14: 'Dimension value.2'
15: 'Dataflow'
16: 'Dataflow name'
17: 'Last update of data'
18: 'Downloaded at'
19: 'Source'
20: 'Source URL'
21: 'Download URL'
22: 'About Inter

inspect the first rows

In [7]:
print("2007-Q1 preview")
display(q1_2007.head(5))

print("2008-Q1 preview")
display(q1_2008.head(5))

print("Bulk preview")
display(bulk.head(5))

2007-Q1 preview


,Publication Table,Publication table description,Period,Dimension,Dimension name,Dimension code,Dimension value,Dimension.1,Dimension name.1,Dimension code.1,Dimension value.1,Dimension.2,Dimension name.2,Dimension code.2,Dimension value.2,Dataflow,Dataflow name,Last update of data,Downloaded at,Source,Source URL,Download URL,About International banking / Consolidated banking statistics
0,BIS:CBS_B4(1.0),Consolidated positions on residents of All countries,2007-Q1,L_MEASURE,Measure,S,Amounts outstanding / Stocks,L_REP_CTY,NaN,5J,NaN,L_CP_COUNTRY,Counterparty country,5J,All countries,BIS:WS_CBS_PUB(1.0),Consolidated banking,31/07/2026,2026-08-25 13:10 CEST,BIS,"https://data.bis.org/topics/CBS/tables-and-dashboards/BIS,CBS_B4,1.0?time_period=2007-Q1","https://data.bis.org/pt_export/BIS,CBS_B4,1.0?file_format=csv&include=code%2Clabel&variant=table...",https://data.bis.org/topics/CBS
1,NaN,Claims,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Other potential exposures (not included in claims) on a guarantor basis,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,Claims on an immediate counterparty basis (F) [1],NaN,NaN,NaN,Risk transfers,NaN,Claims on a guarantor basis (U=F+Q) [1],NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,Total,International,NaN,Local positions in local currencies,Net risk transfers (Q) [1],Of which: outward risk transfers,Total,By sector of counterparty,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,Total,Of which: Up to and including one year,NaN,NaN,NaN,NaN,Banks,Official sector,Non-bank private sector,NaN,Derivatives contracts,Guarantees extended,Credit commitments,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


2008-Q1 preview


,Publication Table,Publication table description,Period,Dimension,Dimension name,Dimension code,Dimension value,Dimension.1,Dimension name.1,Dimension code.1,Dimension value.1,Dimension.2,Dimension name.2,Dimension code.2,Dimension value.2,Dataflow,Dataflow name,Last update of data,Downloaded at,Source,Source URL,Download URL,About International banking / Consolidated banking statistics
0,BIS:CBS_B4(1.0),Consolidated positions on residents of All countries,2008-Q1,L_MEASURE,Measure,S,Amounts outstanding / Stocks,L_REP_CTY,NaN,5J,NaN,L_CP_COUNTRY,Counterparty country,5J,All countries,BIS:WS_CBS_PUB(1.0),Consolidated banking,31/07/2026,2026-08-25 13:34 CEST,BIS,"https://data.bis.org/topics/CBS/tables-and-dashboards/BIS,CBS_B4,1.0?time_period=2008-Q1","https://data.bis.org/pt_export/BIS,CBS_B4,1.0?file_format=csv&include=code%2Clabel&variant=table...",https://data.bis.org/topics/CBS
1,NaN,Claims,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Other potential exposures (not included in claims) on a guarantor basis,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,Claims on an immediate counterparty basis (F) [1],NaN,NaN,NaN,Risk transfers,NaN,Claims on a guarantor basis (U=F+Q) [1],NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,Total,International,NaN,Local positions in local currencies,Net risk transfers (Q) [1],Of which: outward risk transfers,Total,By sector of counterparty,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,Total,Of which: Up to and including one year,NaN,NaN,NaN,NaN,Banks,Official sector,Non-bank private sector,NaN,Derivatives contracts,Guarantees extended,Credit commitments,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Bulk preview


,FREQ,Frequency,L_MEASURE,Measure,L_REP_CTY,Reporting country,CBS_BANK_TYPE,CBS bank type,CBS_BASIS,CBS reporting basis,L_POSITION,Balance sheet position,L_INSTR,Type of instruments,REM_MATURITY,Remaining maturity,CURR_TYPE_BOOK,Currency type of booking location,L_CP_SECTOR,Counterparty sector,L_CP_COUNTRY,Counterparty country,TIME_FORMAT,Time Format,COLLECTION,Collection Indicator,ORG_VISIBILITY,Organisation visibility,Series,1983-Q4,1984-Q2,1984-Q4,1985-Q2,1985-Q4,1986-Q2,1986-Q4,1987-Q2,1987-Q4,1988-Q2,1988-Q4,1989-Q2,1989-Q4,1990-Q2,1990-Q4,1991-Q2,1991-Q4,1992-Q2,1992-Q4,1993-Q2,1993-Q4,...,2013-Q4,2014-Q1,2014-Q2,2014-Q3,2014-Q4,2015-Q1,2015-Q2,2015-Q3,2015-Q4,2016-Q1,2016-Q2,2016-Q3,2016-Q4,2017-Q1,2017-Q2,2017-Q3,2017-Q4,2018-Q1,2018-Q2,2018-Q3,2018-Q4,2019-Q1,2019-Q2,2019-Q3,2019-Q4,2020-Q1,2020-Q2,2020-Q3,2020-Q4,2021-Q1,2021-Q2,2021-Q3,2021-Q4,2022-Q1,2022-Q2,2022-Q3,2022-Q4,2023-Q1,2023-Q2,2023-Q3,2023-Q4,2024-Q1,2024-Q2,2024-Q3,2024-Q4,2025-Q1,2025-Q2,2025-Q3,2025-Q4,2026-Q1
0,Q,Quarterly,S,Amounts outstanding / Stocks,GB,United Kingdom,4R,"Domestic banks(4B), excl. domestic positions",F,Immediate counterparty basis,I,International claims,A,All instruments,A,Total (all maturities),TO1,All currencies,C,Non-financial corporations,5J,All countries,NaN,NaN,E,End of period,E,Public,Q:S:GB:4R:F:I:A:A:TO1:C:5J,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,477336.0,491859.0,493979.0,467274.0,469785.0,448888.0,409017.0,392461.0,382249.0,379322.0,378926.0,374933.0,393285.0,422556.0,436107.0,457321.0,434353.0,403380.0,422104.0,403325.0,408696.0,396572.0,383403.0,402445.0,359913.0,362495.0,367140.0,381166.0,372748.0,394248.0,393563.0,415943.0,349594.0,339561.0,324249.0,349765.0,331914.00,383973.0,365957.0,403277.0,420417.0,422233.0,438012.0,407539.0,412279.0,418161.0,426598.0,449395.0,431520.0
1,Q,Quarterly,S,Amounts outstanding / Stocks,HK,Hong Kong SAR,4B,Domestic banks,U,Guarantor basis,C,Total claims,A,All instruments,A,Total (all maturities),TO1,All currencies,R,Non-bank private sector,VG,British Virgin Islands,NaN,NaN,E,End of period,E,Public,Q:S:HK:4B:U:C:A:A:TO1:R:VG,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Q,Quarterly,B,Break in stocks,SE,Sweden,4R,"Domestic banks(4B), excl. domestic positions",U,Guarantor basis,C,Total claims,A,All instruments,A,Total (all maturities),TO1,All currencies,A,All sectors,GL,Greenland,NaN,NaN,V,Other,E,Public,Q:B:SE:4R:U:C:A:A:TO1:A:GL,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-15.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Q,Quarterly,B,Break in stocks,SE,Sweden,4R,"Domestic banks(4B), excl. domestic positions",U,Guarantor basis,C,Total claims,A,All instruments,A,Total (all maturities),TO1,All currencies,A,All sectors,IR,Iran,NaN,NaN,V,Other,E,Public,Q:B:SE:4R:U:C:A:A:TO1:A:IR,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Q,Quarterly,B,Break in stocks,SE,Sweden,4R,"Domestic banks(4B), excl. domestic positions",U,Guarantor basis,C,Total claims,A,All instruments,A,Total (all maturities),TO1,All currencies,A,All sectors,FO,Faeroe Islands,NaN,NaN,V,Other,E,Public,Q:B:SE:4R:U:C:A:A:TO1:A:FO,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-133.0,N

inspect the last few rows

In [8]:
print("2007-Q1 tail")
display(q1_2007.tail(3))

print("2008-Q1 tail")
display(q1_2008.tail(3))

print("Bulk tail")
display(bulk.tail(3))

2007-Q1 tail


,Publication Table,Publication table description,Period,Dimension,Dimension name,Dimension code,Dimension value,Dimension.1,Dimension name.1,Dimension code.1,Dimension value.1,Dimension.2,Dimension name.2,Dimension code.2,Dimension value.2,Dataflow,Dataflow name,Last update of data,Downloaded at,Source,Source URL,Download URL,About International banking / Consolidated banking statistics
72,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
73,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
74,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


2008-Q1 tail


,Publication Table,Publication table description,Period,Dimension,Dimension name,Dimension code,Dimension value,Dimension.1,Dimension name.1,Dimension code.1,Dimension value.1,Dimension.2,Dimension name.2,Dimension code.2,Dimension value.2,Dataflow,Dataflow name,Last update of data,Downloaded at,Source,Source URL,Download URL,About International banking / Consolidated banking statistics
72,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
73,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
74,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Bulk tail


,FREQ,Frequency,L_MEASURE,Measure,L_REP_CTY,Reporting country,CBS_BANK_TYPE,CBS bank type,CBS_BASIS,CBS reporting basis,L_POSITION,Balance sheet position,L_INSTR,Type of instruments,REM_MATURITY,Remaining maturity,CURR_TYPE_BOOK,Currency type of booking location,L_CP_SECTOR,Counterparty sector,L_CP_COUNTRY,Counterparty country,TIME_FORMAT,Time Format,COLLECTION,Collection Indicator,ORG_VISIBILITY,Organisation visibility,Series,1983-Q4,1984-Q2,1984-Q4,1985-Q2,1985-Q4,1986-Q2,1986-Q4,1987-Q2,1987-Q4,1988-Q2,1988-Q4,1989-Q2,1989-Q4,1990-Q2,1990-Q4,1991-Q2,1991-Q4,1992-Q2,1992-Q4,1993-Q2,1993-Q4,...,2013-Q4,2014-Q1,2014-Q2,2014-Q3,2014-Q4,2015-Q1,2015-Q2,2015-Q3,2015-Q4,2016-Q1,2016-Q2,2016-Q3,2016-Q4,2017-Q1,2017-Q2,2017-Q3,2017-Q4,2018-Q1,2018-Q2,2018-Q3,2018-Q4,2019-Q1,2019-Q2,2019-Q3,2019-Q4,2020-Q1,2020-Q2,2020-Q3,2020-Q4,2021-Q1,2021-Q2,2021-Q3,2021-Q4,2022-Q1,2022-Q2,2022-Q3,2022-Q4,2023-Q1,2023-Q2,2023-Q3,2023-Q4,2024-Q1,2024-Q2,2024-Q3,2024-Q4,2025-Q1,2025-Q2,2025-Q3,2025-Q4,2026-Q1
228367,Q,Quarterly,S,Amounts outstanding / Stocks,BE,Belgium,4R,"Domestic banks(4B), excl. domestic positions",U,Guarantor basis,C,Total claims,A,All instruments,A,Total (all maturities),TO1,All currencies,A,All sectors,NI,Nicaragua,NaN,NaN,E,End of period,E,Public,Q:S:BE:4R:U:C:A:A:TO1:A:NI,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,7.0,6.0,6.0,6.0,6.0,5.0,6.0,5.0,5.0,5.0,4.0,4.0,3.0,3.0,4.0,3.0,3.0,2.0,2.0,2.0,2.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,NaN,NaN,NaN,NaN,NaN,1.0,1.0
228368,Q,Quarterly,S,Amounts outstanding / Stocks,BE,Belgium,4R,"Domestic banks(4B), excl. domestic positions",U,Guarantor basis,C,Total claims,A,All instruments,A,Total (all maturities),TO1,All currencies,A,All sectors,NE,Niger,NaN,NaN,E,End of period,E,Public,Q:S:BE:4R:U:C:A:A:TO1:A:NE,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,1.0,1.0,1.0,1.0,1.0,NaN,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,1.0,1.0,NaN,1.0,1.0,2.0,NaN,NaN,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
228369,Q,Quarterly,S,Amounts outstanding / Stocks,BE,Belgium,4R,"Domestic banks(4B), excl. domestic positions",U,Guarantor basis,C,Total claims,A,All instruments,A,Total (all maturities),TO1,All currencies,A,All sectors,MW,Malawi,NaN,NaN,E,End of period,E,Public,Q:S:BE:4R:U:C:A:A:TO1:A:MW,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


inspect data types

In [9]:
print("2007-Q1 data types")
display(q1_2007.dtypes.to_frame("dtype"))

print("2008-Q1 data types")
display(q1_2008.dtypes.to_frame("dtype"))

print("Bulk data types")
display(bulk.dtypes.to_frame("dtype"))

2007-Q1 data types


,dtype
Publication Table,object
Publication table description,object
Period,object
Dimension,object
Dimension name,object
Dimension code,object
Dimension value,object
Dimension.1,object
Dimension name.1,object
Dimension code.1,object


2008-Q1 data types


,dtype
Publication Table,object
Publication table description,object
Period,object
Dimension,object
Dimension name,object
Dimension code,object
Dimension value,object
Dimension.1,object
Dimension name.1,object
Dimension code.1,object


Bulk data types


,dtype
FREQ,object
Frequency,object
L_MEASURE,object
Measure,object
L_REP_CTY,object
...,...
2025-Q1,float64
2025-Q2,float64
2025-Q3,float64
2025-Q4,float64


check duplicate column names

In [10]:
def duplicate_columns(df, name):
    duplicated = df.columns[df.columns.duplicated()].tolist()
    print(f"{name} duplicate columns:", duplicated)

duplicate_columns(q1_2007, "2007-Q1")
duplicate_columns(q1_2008, "2008-Q1")
duplicate_columns(bulk, "Bulk")

2007-Q1 duplicate columns: []
2008-Q1 duplicate columns: []
Bulk duplicate columns: []


check missing values

In [11]:
def missing_report(df, name):
    report = pd.DataFrame({
        "missing_count": df.isna().sum(),
        "missing_percent": (df.isna().mean() * 100).round(2),
        "dtype": df.dtypes.astype(str)
    })
    report = report.sort_values("missing_count", ascending=False)
    
    print(f"\nMissing values: {name}")
    display(report)

missing_report(q1_2007, "2007-Q1")
missing_report(q1_2008, "2008-Q1")
missing_report(bulk, "Bulk dataset")


Missing values: 2007-Q1


,missing_count,missing_percent,dtype
Dataflow name,74,98.67,object
Last update of data,74,98.67,object
Downloaded at,74,98.67,object
Source,74,98.67,object
Source URL,74,98.67,object
Download URL,74,98.67,object
About International banking / Consolidated banking statistics,74,98.67,object
Dataflow,74,98.67,object
Dimension.2,72,96.00,object
Dimension value,30,40.00,object



Missing values: 2008-Q1


,missing_count,missing_percent,dtype
Dataflow name,74,98.67,object
Last update of data,74,98.67,object
Downloaded at,74,98.67,object
Source,74,98.67,object
Source URL,74,98.67,object
Download URL,74,98.67,object
About International banking / Consolidated banking statistics,74,98.67,object
Dataflow,74,98.67,object
Dimension.2,72,96.00,object
Dimension value,30,40.00,object



Missing values: Bulk dataset


,missing_count,missing_percent,dtype
TIME_FORMAT,228370,100.00,float64
Time Format,228370,100.00,float64
1984-Q2,218310,95.59,float64
1983-Q4,218224,95.56,float64
1984-Q4,218074,95.49,float64
...,...,...,...
CURR_TYPE_BOOK,0,0.00,object
L_CP_SECTOR,0,0.00,object
Currency type of booking location,0,0.00,object
Counterparty sector,0,0.00,object


check unique values in text columns

In [12]:
def categorical_summary(df, name, max_unique=30):
    print(f"\nCategorical summary: {name}")
    print("=" * 30)
    
    for col in df.columns:
        n_unique = df[col].nunique(dropna=False)
        if df[col].dtype == "object" or n_unique <= max_unique:
            print(f"\nColumn: {col}")
            print(f"Unique values: {n_unique}")
            display(df[col].value_counts(dropna=False).head(30))

categorical_summary(q1_2007, "2007-Q1")
categorical_summary(q1_2008, "2008-Q1")


Categorical summary: 2007-Q1

Column: Publication Table
Unique values: 33


Publication Table
NaN                                         14
    Japan                                    2
Foreign banks                                2
  Of which: parents in CBS rep countries     2
    Australia                                2
    Austria                                  2
    Belgium                                  2
    Brazil                                   2
    Canada                                   2
    Chile                                    2
    Chinese Taipei                           2
    Finland                                  2
    France                                   2
    Germany                                  2
    Greece                                   2
    Ireland                                  2
    Italy                                    2
    Switzerland                              2
    Korea                                    2
    Mexico                                   2
    Netherlands                           


Column: Publication table description
Unique values: 57


Publication table description
NaN                                                     16
Claims on an immediate counterparty basis (F) [1]        2
Claims                                                   2
Total                                                    2
Consolidated positions on residents of All countries     1
26,469,809.0                                             1
25,858,237.0                                             1
356,210.0                                                1
455,193.0                                                1
1,235,425.0                                              1
43,019.0                                                 1
641,477.0                                                1
2,958.0                                                  1
151,619.0                                                1
6,170.0                                                  1
2,954,444.0                                              1
3,835,251.0               


Column: Period
Unique values: 56


...

Source URL
NaN                                                                                         74
https://data.bis.org/topics/CBS/tables-and-dashboards/BIS,CBS_B4,1.0?time_period=2008-Q1     1
Name: count, dtype: int64


Column: Download URL
Unique values: 2


Download URL
NaN                                                                                                                     74
https://data.bis.org/pt_export/BIS,CBS_B4,1.0?file_format=csv&include=code%2Clabel&variant=table&time_period=2008-Q1     1
Name: count, dtype: int64


Column: About International banking / Consolidated banking statistics
Unique values: 2


About International banking / Consolidated banking statistics
NaN                                74
https://data.bis.org/topics/CBS     1
Name: count, dtype: int64

In [13]:
print("Bulk unique-value counts")
display(
    pd.DataFrame({
        "dtype": bulk.dtypes.astype(str),
        "n_unique": bulk.nunique(dropna=False),
        "missing_count": bulk.isna().sum()
    }).sort_values("n_unique")
)

Bulk unique-value counts


,dtype,n_unique,missing_count
FREQ,object,1,0
Frequency,object,1,0
Time Format,float64,1,228370
TIME_FORMAT,float64,1,228370
Organisation visibility,object,1,0
...,...,...,...
2016-Q4,float64,27994,141744
2017-Q4,float64,29295,141455
2017-Q2,float64,29324,138262
2017-Q1,float64,30400,133781


identify possible date columns

In [14]:
def possible_date_columns(df):
    keywords = ["date", "time", "period", "quarter", "year", "reference"]
    return [
        col for col in df.columns
        if any(keyword in str(col).lower() for keyword in keywords)
    ]

print("Possible date columns in 2007-Q1:", possible_date_columns(q1_2007))
print("Possible date columns in 2008-Q1:", possible_date_columns(q1_2008))
print("Possible date columns in bulk:", possible_date_columns(bulk))

Possible date columns in 2007-Q1: ['Period', 'Last update of data', 'About International banking / Consolidated banking statistics']
Possible date columns in 2008-Q1: ['Period', 'Last update of data', 'About International banking / Consolidated banking statistics']
Possible date columns in bulk: ['TIME_FORMAT', 'Time Format']


date/period columns

In [16]:
# Actual date/period column in the quarterly files
print("2007-Q1 periods:")
print(q1_2007["Period"].value_counts(dropna=False))

print("\n2008-Q1 periods:")
print(q1_2008["Period"].value_counts(dropna=False))

2007-Q1 periods:
Period
NaN                                                       18
International                                              2
Total                                                      2
2007-Q1                                                    1
16,755,762.0                                               1
16,144,681.0                                               1
113,881.0                                                  1
328,091.0                                                  1
839,325.0                                                  1
41,298.0                                                   1
290,869.0                                                  1
2,958.0                                                    1
136,598.0                                                  1
6,170.0                                                    1
1,845,280.0                                                1
3,170,251.0                                                1


In [17]:
print("Bulk columns:")
print(bulk.columns.tolist())

Bulk columns:
['FREQ', 'Frequency', 'L_MEASURE', 'Measure', 'L_REP_CTY', 'Reporting country', 'CBS_BANK_TYPE', 'CBS bank type', 'CBS_BASIS', 'CBS reporting basis', 'L_POSITION', 'Balance sheet position', 'L_INSTR', 'Type of instruments', 'REM_MATURITY', 'Remaining maturity', 'CURR_TYPE_BOOK', 'Currency type of booking location', 'L_CP_SECTOR', 'Counterparty sector', 'L_CP_COUNTRY', 'Counterparty country', 'TIME_FORMAT', 'Time Format', 'COLLECTION', 'Collection Indicator', 'ORG_VISIBILITY', 'Organisation visibility', 'Series', '1983-Q4', '1984-Q2', '1984-Q4', '1985-Q2', '1985-Q4', '1986-Q2', '1986-Q4', '1987-Q2', '1987-Q4', '1988-Q2', '1988-Q4', '1989-Q2', '1989-Q4', '1990-Q2', '1990-Q4', '1991-Q2', '1991-Q4', '1992-Q2', '1992-Q4', '1993-Q2', '1993-Q4', '1994-Q2', '1994-Q4', '1995-Q2', '1995-Q4', '1996-Q2', '1996-Q4', '1997-Q2', '1997-Q4', '1998-Q2', '1998-Q4', '1999-Q2', '1999-Q4', '2000-Q1', '2000-Q2', '2000-Q3', '2000-Q4', '2001-Q1', '2001-Q2', '2001-Q3', '2001-Q4', '2002-Q1', '2002-

print("Bulk columns:")
print(bulk.columns.tolist())

In [15]:
def possible_country_columns(df):
    keywords = [
        "country", "economy", "reporter", "reporting",
        "counterparty", "nationality", "location", "borrower", "lender"
    ]
    return [
        col for col in df.columns
        if any(keyword in str(col).lower() for keyword in keywords)
    ]

print("Possible country columns in 2007-Q1:")
print(possible_country_columns(q1_2007))

print("\nPossible country columns in 2008-Q1:")
print(possible_country_columns(q1_2008))

print("\nPossible country columns in bulk:")
print(possible_country_columns(bulk))

Possible country columns in 2007-Q1:
[]

Possible country columns in 2008-Q1:
[]

Possible country columns in bulk:
['Reporting country', 'CBS reporting basis', 'Currency type of booking location', 'Counterparty sector', 'L_CP_COUNTRY', 'Counterparty country']


In [18]:
country_col_2007 = q1_2007.columns[0]
country_col_2008 = q1_2008.columns[0]

print("2007-Q1 country column:", country_col_2007)
print("2008-Q1 country column:", country_col_2008)

print("\n2007-Q1 countries/economies:")
print(q1_2007[country_col_2007].dropna().unique())

print("\n2008-Q1 countries/economies:")
print(q1_2008[country_col_2008].dropna().unique())

2007-Q1 country column: Publication Table
2008-Q1 country column: Publication Table

2007-Q1 countries/economies:
['BIS:CBS_B4(1.0)' 'Foreign banks'
 '  Of which: parents in CBS rep countries' '    Australia' '    Austria'
 '    Belgium' '    Brazil' '    Canada' '    Chile' '    Chinese Taipei'
 '    Finland' '    France' '    Germany' '    Greece' '    Ireland'
 '    Italy' '    Japan' '    Korea' '    Mexico' '    Netherlands'
 '    Panama' '    Portugal' '    Spain' '    Sweden' '    Switzerland'
 '    Türkiye' '    United Kingdom' '    United States'
 '  Memo: Domestic banks  [2]' '    Worldwide offices (consolidated)'
 '[1] For foreign banks and banks with parents in CBS reporting countries, F plus Q does not sum to U because F is reported by a larger sample of banks; for the latest quarter, immediate counterparty data are reported by banks in 31 countries and guarantor basis data by banks in 26. For parents in individual CBS reporting countries, F plus Q may not sum to U because

counterparty country

In [19]:
country_col_bulk = "Counterparty country"

print("Countries in bulk:")
print(bulk[country_col_bulk].dropna().unique())

Countries in bulk:
['All countries' 'British Virgin Islands' 'Greenland' 'Iran'
 'Faeroe Islands' 'Senegal' 'El Salvador' 'Sint Maarten' 'Chad' 'Thailand'
 'Tunisia' 'Türkiye' 'Serbia' 'Russia' 'Saudi Arabia' 'Seychelles'
 'Sweden' 'Singapore' 'Slovenia' 'Slovakia'
 'St Vincent and the Grenadines' 'Venezuela' 'Vietnam' 'Yemen'
 'South Africa' 'Zambia' 'Zimbabwe' 'Trinidad and Tobago' 'Chinese Taipei'
 'Tanzania' 'Ukraine' 'Uganda' 'United States' 'Uruguay' 'Uzbekistan'
 'St Kitts and Nevis' 'North Korea' 'Mongolia'
 'Unallocated British Overseas Territories' 'Burundi'
 'Bonaire, Sint Eustatius and Saba' 'Bhutan' 'Eritrea' 'Ethiopia' 'Fiji'
 'Afghanistan' 'Anguilla' 'Italy' 'Mauritius' 'Jersey' 'Kyrgyz Republic'
 'Jamaica' 'Kiribati' 'Mexico' 'Jordan' 'Malaysia' 'Japan' 'Kenya'
 'Namibia' 'Cambodia' 'Korea' 'Nigeria' 'Hungary' 'Montenegro' 'Indonesia'
 'Ireland' 'Marshall Islands' 'Israel' 'North Macedonia' 'Isle of Man'
 'India' 'Iraq' 'Macao SAR' 'Iceland' 'Malta' 'Liberia' 'Lithuania

country reporting the banking statistics

In [21]:
reporting_country_col_bulk = "Reporting country"

print("Reporting countries in bulk:")
print(bulk[reporting_country_col_bulk].dropna().unique())

Reporting countries in bulk:
['United Kingdom' 'Hong Kong SAR' 'Sweden' 'Austria' 'Netherlands'
 'Ireland' 'France' 'Chinese Taipei' 'Brazil' 'Norway' 'Greece' 'Canada'
 'All reporting countries' 'Italy' 'Türkiye' 'Australia' 'Finland'
 'Switzerland' 'Singapore' 'United States' 'Malaysia' 'Denmark' 'Portugal'
 'Luxembourg' 'Germany' 'Japan' 'Chile' 'Spain' 'India' 'Belgium' 'Korea'
 'Mexico' 'Panama']


compare the schemas of the three files

In [22]:
cols_2007 = set(q1_2007.columns)
cols_2008 = set(q1_2008.columns)
cols_bulk = set(bulk.columns)

print("Columns in 2007-Q1 but not 2008-Q1:")
print(sorted(cols_2007 - cols_2008))

print("\nColumns in 2008-Q1 but not 2007-Q1:")
print(sorted(cols_2008 - cols_2007))

print("\nColumns in 2007-Q1 but not bulk:")
print(sorted(cols_2007 - cols_bulk))

print("\nColumns in bulk but not 2007-Q1:")
print(sorted(cols_bulk - cols_2007))

print("\nExact 2007-Q1 and 2008-Q1 schemas:", cols_2007 == cols_2008)
print("Exact 2007-Q1 and bulk schemas:", cols_2007 == cols_bulk)

Columns in 2007-Q1 but not 2008-Q1:
[]

Columns in 2008-Q1 but not 2007-Q1:
[]

Columns in 2007-Q1 but not bulk:
['About International banking / Consolidated banking statistics', 'Dataflow', 'Dataflow name', 'Dimension', 'Dimension code', 'Dimension code.1', 'Dimension code.2', 'Dimension name', 'Dimension name.1', 'Dimension name.2', 'Dimension value', 'Dimension value.1', 'Dimension value.2', 'Dimension.1', 'Dimension.2', 'Download URL', 'Downloaded at', 'Last update of data', 'Period', 'Publication Table', 'Publication table description', 'Source', 'Source URL']

Columns in bulk but not 2007-Q1:
['1983-Q4', '1984-Q2', '1984-Q4', '1985-Q2', '1985-Q4', '1986-Q2', '1986-Q4', '1987-Q2', '1987-Q4', '1988-Q2', '1988-Q4', '1989-Q2', '1989-Q4', '1990-Q2', '1990-Q4', '1991-Q2', '1991-Q4', '1992-Q2', '1992-Q4', '1993-Q2', '1993-Q4', '1994-Q2', '1994-Q4', '1995-Q2', '1995-Q4', '1996-Q2', '1996-Q4', '1997-Q2', '1997-Q4', '1998-Q2', '1998-Q4', '1999-Q2', '1999-Q4', '2000-Q1', '2000-Q2', '2000-Q3

Check whether the trial files contain one date or multiple dates

In [23]:
def inspect_date_like_values(df, name):
    print(f"\n{name}")
    for col in df.columns:
        values = df[col].dropna().astype(str)
        sample = values.head(100).str.cat(sep=" ")
        date_signal = any(
            term in sample.lower()
            for term in ["2007", "2008", "2009", "2010", "q1", "q2", "q3", "q4"]
        )
        if date_signal:
            print(f"\nColumn: {col}")
            print(values.value_counts().head(20))

inspect_date_like_values(q1_2007, "2007-Q1")
inspect_date_like_values(q1_2008, "2008-Q1")


2007-Q1

Column: Publication table description
Publication table description
Claims                                                  2
Claims on an immediate counterparty basis (F) [1]       2
Total                                                   2
Consolidated positions on residents of All countries    1
26,469,809.0                                            1
25,858,237.0                                            1
356,210.0                                               1
455,193.0                                               1
1,235,425.0                                             1
43,019.0                                                1
641,477.0                                               1
2,958.0                                                 1
151,619.0                                               1
6,170.0                                                 1
2,954,444.0                                             1
3,835,251.0                                         

Check numeric columns

In [24]:
def numeric_summary(df, name):
    numeric = df.select_dtypes(include=np.number)
    print(f"\nNumeric columns in {name}:")
    print(list(numeric.columns))
    
    if not numeric.empty:
        display(numeric.describe().T)

numeric_summary(q1_2007, "2007-Q1")
numeric_summary(q1_2008, "2008-Q1")


Numeric columns in 2007-Q1:
[]

Numeric columns in 2008-Q1:
[]


In [25]:
bulk_numeric = bulk.select_dtypes(include=np.number)

print("Number of numeric columns in bulk:", bulk_numeric.shape[1])
display(bulk_numeric.describe().T)

Number of numeric columns in bulk: 140


,count,mean,std,min,25%,50%,75%,max
TIME_FORMAT,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Time Format,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1983-Q4,10146.0,4859.390794,3.442339e+04,1.000,15.000,94.000,758.0000,7.548150e+05
1984-Q2,10060.0,5024.274851,3.537404e+04,1.000,17.000,104.000,761.0000,7.748060e+05
1984-Q4,10296.0,5364.629371,3.727056e+04,1.000,18.000,104.000,824.0000,7.774580e+05
...,...,...,...,...,...,...,...,...
2025-Q1,77612.0,133020.729093,1.811588e+06,-701057.507,2.540,152.000,3255.2365,1.109580e+08
2025-Q2,75409.0,145440.255155,1.948206e+06,-726475.490,4.827,196.129,3874.0000,1.170356e+08
2025-Q3,74266.0,149101.201362,1.974300e+06,-749570.792,4.000,193.293,4047.8170,1.176059e+08
2025-Q4,72198.0,153726.563932,2.004166e+06,-739733.550,6.208,236.666,4635.9465,1.173006e+08


Basic memory and file-size information

In [26]:
def memory_report(df, name):
    memory_mb = df.memory_usage(deep=True).sum() / 1024**2
    print(f"{name}: {memory_mb:.2f} MB in memory")
    print(f"Rows: {len(df):,}")
    print(f"Columns: {len(df.columns):,}")

memory_report(q1_2007, "2007-Q1")
memory_report(q1_2008, "2008-Q1")
memory_report(bulk, "Bulk dataset")

2007-Q1: 0.09 MB in memory
Rows: 75
Columns: 23
2008-Q1: 0.09 MB in memory
Rows: 75
Columns: 23
Bulk dataset: 588.09 MB in memory
Rows: 228,370
Columns: 167


## Project Data Structure

The project keeps original downloads unchanged in the `data/raw/` directory.

- `data/raw/` contains the original BIS files exactly as downloaded.
- `data/processed/` will contain cleaned and filtered analysis-ready datasets created in this notebook.
- `notebook/` contains the code and documentation used to transform the raw data into analysis-ready files.

The raw files will not be overwritten. All transformations will be reproducible in code, so that the team can trace each processed observation back to its original BIS source.
```text
data/
├── raw/
│   ├── consolidated banking statistics.csv
│   ├── 2007-Q1.csv
│   └── 2008-Q1.csv
└── processed/
```

## Initial Data Inspection

The first stage of preprocessing examines the structure of each file rather than immediately analyzing banking exposures.

For each downloaded dataset, we inspect:

- File availability and loading success.
- Number of rows and columns.
- Column names and data types.
- Preview rows and table layout.
- Missing values.
- Numeric columns.
- Date or quarter fields.
- Country, measure, and reporting-basis fields.

This inspection is necessary because BIS provides both formatted dashboard exports and bulk statistical data. Files can describe the same topic while having different structures and intended uses.

## Comparison of the Downloaded Files

The three files refer to the BIS Consolidated Banking Statistics but serve different purposes.

| File | Shape after loading | Structure | Intended use |
|---|---:|---|---|
| `2007-Q1.csv` | 75 rows × 23 columns | Formatted dashboard/table export; metadata and display labels stored mainly as text | Documents the selected BIS Table B4 view and its 2007-Q1 dashboard settings |
| `2008-Q1.csv` | 75 rows × 23 columns | Formatted dashboard/table export; metadata and display labels stored mainly as text | Documents the selected BIS Table B4 view and its 2008-Q1 dashboard settings |
| `consolidated banking statistics.csv` | 228,370 rows × 167 columns | Structured BIS bulk dataset; descriptive dimensions plus one numeric column for each quarter | Main source for filtering, analysis, network construction, and time-series comparison |

The two Table B4 exports contain columns such as `Publication Table`, `Period`, `Dimension`, `Dimension name`, and `Dimension value`. They are formatted to reproduce a dashboard table rather than to provide a tidy country-to-country analytical dataset. Consequently, pandas imports their fields as text (`object`) and does not identify conventional numeric value columns.

By contrast, the bulk dataset contains structured dimensions such as reporting country, counterparty country, measure, reporting basis, balance-sheet position, counterparty sector, and quarterly observation columns. It includes numeric quarterly columns from the historical period through 2026-Q1, including all quarters from 2007-Q1 through 2010-Q4.

## Why the Table B4 Exports Do Not Show Numeric Columns

The absence of numeric columns in the 2007-Q1 and 2008-Q1 dashboard-export files does not mean that BIS data are missing.

These files are presentation-oriented exports. They include dashboard headings, table sections, labels, notes, metadata, and download information. For example, rows may describe headings such as `Claims`, `Claims on an immediate counterparty basis`, `International`, or `Claims on a guarantor basis`.

This format is useful for verifying what was displayed in the BIS dashboard at a selected date. However, it is not well suited to direct statistical analysis, network construction, or comparison across many quarters.

The bulk download stores the underlying observations in an analysis-friendly format: each row identifies a banking series, while quarter columns such as `2007-Q1`, `2008-Q1`, and `2010-Q4` contain the numerical values.

## Data-Source Decision

The project needs a consistent country-to-country banking network for the crisis period. The main dataset must therefore allow the team to identify:

- The reporting country, representing the banking system providing the claims.
- The counterparty country, representing the country receiving the exposure.
- The type of exposure or claim being measured.
- The reporting basis used for the exposure.
- The sector and other relevant dimensions.
- A numerical observation for each quarter from 2007 to 2010.

The BIS bulk download is the appropriate main source because it contains these fields and the required quarterly values in one file.

The Table B4 dashboard exports will be retained as validation and documentation sources. They will help confirm the selected dashboard table, selected date, measure, source URL, terminology, and relevant dashboard configuration. They can also be used to cross-check selected aggregate values against the equivalent filtered observations in the bulk dataset.

## Identifying the Relevant Banking Series

The bulk download contains many BIS Consolidated Banking Statistics series, not one single network.

For example, it includes different combinations of:

- Amounts outstanding, changes, breaks, or other measures.
- Reporting countries.
- Domestic-bank and foreign-bank categories.
- Immediate-counterparty and guarantor reporting bases.
- Total claims, international claims, local positions, and other balance-sheet positions.
- Different instrument types.
- Different maturity categories.
- Different currencies.
- Different counterparty sectors.
- Different counterparty countries.

Therefore, the next step is not to delete data randomly. It is to identify the exact combination of BIS dimension codes and labels that corresponds to the project’s chosen definition of a cross-border banking edge.

For the initial network, the team will likely select a consistent series definition based on:

- `Measure`: Amounts outstanding / Stocks.
- `CBS reporting basis`: Immediate counterparty basis.
- `Balance sheet position`: A claims measure appropriate to cross-border exposure, likely international claims or total claims depending on the dashboard definition.
- `Type of instruments`: All instruments.
- `Remaining maturity`: Total (all maturities).
- `Currency type of booking location`: All currencies.
- `Counterparty sector`: All sectors.
- `Reporting country`: Individual reporting banking systems.
- `Counterparty country`: Individual borrower/counterparty countries.

The exact codes and labels will be confirmed by examining the values available in each relevant bulk-data column before filtering.

## Planned Preprocessing Steps

The preprocessing workflow will proceed in the following order:

1. Inspect the unique codes and labels in the bulk dataset's key dimensions.
2. Select one consistent definition of the lending/exposure series for the network.
3. Filter the bulk data to the crisis-analysis window: 2007-Q1 through 2010-Q4.
4. Retain only rows with an identified reporting country, counterparty country, and numerical observation.
5. Reshape the selected quarter columns from wide format to long format, producing one row per reporting country, counterparty country, and quarter.
6. Check for missing values, duplicated observations, aggregate-country rows, and non-country categories.
7. Validate selected observations or totals against the 2007-Q1 and 2008-Q1 Table B4 dashboard exports.
8. Save the resulting analysis-ready dataset in `data/processed/`.
9. Use the processed dataset in later notebooks for network mapping, causal analysis, strategic-behavior analysis, and group integration.

No missing values will be replaced with zero until the team has determined whether each missing value represents a true absence of reporting, an unavailable observation, confidentiality protection, or a non-applicable series.

## Expected Processed Dataset

The analysis-ready dataset will be saved in `data/processed/` and will use a long, network-ready structure.

Each row will represent one directed country-to-country exposure in one quarter:

| period | reporting_country | counterparty_country | claim_value_usd_millions | reporting_basis | position | sector |
|---|---|---|---:|---|---|---|
| 2007-Q1 | Reporting country | Counterparty country | BIS value | Immediate counterparty basis | Selected claims measure | All sectors |

This structure will make it possible to:

- Rank countries by lending connections and exposure.
- Identify hubs, peripheral countries, and potential bridge countries.
- Track lending changes across the 2008–09 period.
- Visualize directed and weighted networks.
- Examine concentrated exposures and possible cascade paths.
- Compare selected observations with dashboard exports.

## Conclusion of the Initial Data Review

The initial review supports using `consolidated banking statistics.csv` as the primary dataset for this project. Unlike the two small Table B4 exports, which are formatted dashboard representations containing mainly labels and metadata, the bulk file contains structured reporting-country and counterparty-country dimensions together with numerical quarterly values. It includes all required crisis-period quarters from 2007-Q1 through 2010-Q4. The project will therefore retain the Table B4 exports for dashboard documentation and validation, while using the bulk dataset for reproducible filtering, cleaning, time-series analysis, and network construction. The next task is to inspect the bulk dataset’s dimension codes and labels, choose one consistent definition of cross-border lending exposure, and then create a filtered analysis-ready dataset in `data/processed/`.

# ===============================================================================

# Step 1: Inspect key dimension values

In [27]:
key_dimensions = [
    "Measure",
    "CBS reporting basis",
    "Balance sheet position",
    "CBS bank type",
    "Type of instruments",
    "Remaining maturity",
    "Currency type of booking location",
    "Counterparty sector",
]

for column in key_dimensions:
    print("\n" + "=" * 90)
    print(f"{column}")
    print("=" * 90)
    
    if column not in bulk.columns:
        print(f"Column not found: {column}")
        continue
    
    summary = (
        bulk[[column]]
        .value_counts(dropna=False)
        .reset_index(name="row_count")
    )
    
    display(summary)


Measure


,Measure,row_count
0,Amounts outstanding / Stocks,179016
1,Break in stocks,49354



CBS reporting basis


,CBS reporting basis,row_count
0,Immediate counterparty basis,108574
1,Guarantor basis,85989
2,Net risk transfers (Inward-Outward),18243
3,Outward risk transfers,15564



Balance sheet position


...

,Remaining maturity,row_count
0,Total (all maturities),199329
1,Up to and including 1 year,24367
2,Over 2 years,2386
3,Over 1 year and up to and including 2 years,2288



Currency type of booking location


,Currency type of booking location,row_count
0,All currencies,217775
1,Local currency,10595



Counterparty sector


,Counterparty sector,row_count
0,All sectors,172019
1,Non-bank private sector,15866
2,"Banks, total",12590
3,Official sector,10289
4,Non-bank financial institutions,8499
5,Households and NPISHs,3108
6,Non-financial corporations,3039
7,Non-financial private sector,2960


In [28]:
code_label_pairs = [
    ("L_MEASURE", "Measure"),
    ("CBS_BASIS", "CBS reporting basis"),
    ("L_POSITION", "Balance sheet position"),
    ("CBS_BANK_TYPE", "CBS bank type"),
    ("L_INSTR", "Type of instruments"),
    ("REM_MATURITY", "Remaining maturity"),
    ("CURR_TYPE_BOOK", "Currency type of booking location"),
    ("L_CP_SECTOR", "Counterparty sector"),
]

for code_col, label_col in code_label_pairs:
    print("\n" + "=" * 90)
    print(f"{label_col}: code-to-label mapping")
    print("=" * 90)
    
    mapping = (
        bulk[[code_col, label_col]]
        .drop_duplicates()
        .sort_values([label_col, code_col], na_position="last")
        .reset_index(drop=True)
    )
    
    display(mapping)


Measure: code-to-label mapping


,L_MEASURE,Measure
0,S,Amounts outstanding / Stocks
1,B,Break in stocks



CBS reporting basis: code-to-label mapping


,CBS_BASIS,CBS reporting basis
0,U,Guarantor basis
1,F,Immediate counterparty basis
2,Q,Net risk transfers (Inward-Outward)
3,O,Outward risk transfers



Balance sheet position: code-to-label mapping


...

,REM_MATURITY,Remaining maturity
0,M,Over 1 year and up to and including 2 years
1,N,Over 2 years
2,A,Total (all maturities)
3,U,Up to and including 1 year



Currency type of booking location: code-to-label mapping


,CURR_TYPE_BOOK,Currency type of booking location
0,TO1,All currencies
1,LC1,Local currency



Counterparty sector: code-to-label mapping


,L_CP_SECTOR,Counterparty sector
0,A,All sectors
1,B,"Banks, total"
2,H,Households and NPISHs
3,F,Non-bank financial institutions
4,R,Non-bank private sector
5,C,Non-financial corporations
6,S,Non-financial private sector
7,O,Official sector


# Step 2: Apply the initial BIS network filters

In [29]:
# Exact filters selected from the BIS code-to-label mappings

selected_filters = {
    "L_MEASURE": "S",          # Amounts outstanding / Stocks
    "CBS_BASIS": "F",          # Immediate counterparty basis
    "L_POSITION": "I",        # International claims
    "CBS_BANK_TYPE": "4B",    # Domestic banks
    "L_INSTR": "A",            # All instruments
    "REM_MATURITY": "A",       # Total (all maturities)
    "CURR_TYPE_BOOK": "TO1",  # All currencies
    "L_CP_SECTOR": "A",       # All sectors
}

filtered_bulk = bulk.copy()

for column, code in selected_filters.items():
    filtered_bulk = filtered_bulk[filtered_bulk[column] == code]

print("Filtered dataset shape:", filtered_bulk.shape)

print("\nRemaining rows by selected dimension:")
for column, code in selected_filters.items():
    label_column = {
        "L_MEASURE": "Measure",
        "CBS_BASIS": "CBS reporting basis",
        "L_POSITION": "Balance sheet position",
        "CBS_BANK_TYPE": "CBS bank type",
        "L_INSTR": "Type of instruments",
        "REM_MATURITY": "Remaining maturity",
        "CURR_TYPE_BOOK": "Currency type of booking location",
        "L_CP_SECTOR": "Counterparty sector",
    }[column]
    
    if label_column in filtered_bulk.columns:
        print(f"{label_column}:")
        display(filtered_bulk[label_column].value_counts(dropna=False))

print("\nPreview of filtered rows:")
display(
    filtered_bulk[
        [
            "Series",
            "Reporting country",
            "Counterparty country",
            "Measure",
            "CBS reporting basis",
            "Balance sheet position",
            "CBS bank type",
            "Type of instruments",
            "Remaining maturity",
            "Currency type of booking location",
            "Counterparty sector",
        ]
    ].head(10)
)

Filtered dataset shape: (6514, 167)

Remaining rows by selected dimension:
Measure:


Measure
Amounts outstanding / Stocks    6514
Name: count, dtype: int64

CBS reporting basis:


CBS reporting basis
Immediate counterparty basis    6514
Name: count, dtype: int64

Balance sheet position:


...

Currency type of booking location
All currencies    6514
Name: count, dtype: int64

Counterparty sector:


Counterparty sector
All sectors    6514
Name: count, dtype: int64


Preview of filtered rows:


,Series,Reporting country,Counterparty country,Measure,CBS reporting basis,Balance sheet position,CBS bank type,Type of instruments,Remaining maturity,Currency type of booking location,Counterparty sector
1098,Q:S:GR:4B:F:I:A:A:TO1:A:2Z,Greece,Unallocated West Indies UK,Amounts outstanding / Stocks,Immediate counterparty basis,International claims,Domestic banks,All instruments,Total (all maturities),All currencies,All sectors
1122,Q:S:GR:4B:F:I:A:A:TO1:A:3P,Greece,All countries excluding residents,Amounts outstanding / Stocks,Immediate counterparty basis,International claims,Domestic banks,All instruments,Total (all maturities),All currencies,All sectors
1152,Q:S:GR:4B:F:I:A:A:TO1:A:1C,Greece,International organisations,Amounts outstanding / Stocks,Immediate counterparty basis,International claims,Domestic banks,All instruments,Total (all maturities),All currencies,All sectors
1156,Q:S:GR:4B:F:I:A:A:TO1:A:1E,Greece,Residents/Local,Amounts outstanding / Stocks,Immediate counterparty basis,International claims,Domestic banks,All instruments,Total (all maturities),All currencies,All sectors
1215,Q:S:GR:4B:F:I:A:A:TO1:A:1W,Greece,Unallocated British Overseas Territories,Amounts outstanding / Stocks,Immediate counterparty basis,International claims,Domestic banks,All instruments,Total (all maturities),All currencies,All sectors
1217,Q:S:GR:4B:F:I:A:A:TO1:A:BI,Greece,Burundi,Amounts outstanding / Stocks,Immediate counterparty basis,International claims,Domestic banks,All instruments,Total (all maturities),All currencies,All sectors
1218,Q:S:GR:4B:F:I:A:A:TO1:A:ET,Greece,Ethiopia,Amounts outstanding / Stocks,Immediate counterparty basis,International claims,Domestic banks,All instruments,Total (all maturities),All currencies,All sectors
1219,Q:S:GR:4B:F:I:A:A:TO1:A:FJ,Greece,Fiji,Amounts outstanding / Stocks,Immediate counterparty basis,International claims,Domestic banks,All instruments,Total (all maturities),All currencies,All sectors
1220,Q:S:GR:4B:F:I:A:A:TO1:A:GL,Greece,Greenland,Amounts outstanding / Stocks,Immediate counterparty basis,International claims,Domestic banks,All instruments,Total (all maturities),All currencies,All sectors
1222,Q:S:GR:4B:F:I:A:A:TO1:A:IR,Greece,Iran,Amounts outstanding / Stocks,Immediate counterparty basis,International claims,Domestic banks,All instruments,Total (all maturities),All currencies,All sectors


inspect counterparty categories

In [30]:
counterparty_summary = (
    filtered_bulk["Counterparty country"]
    .value_counts(dropna=False)
    .rename_axis("counterparty_country")
    .reset_index(name="series_count")
)

print("Number of unique counterparty categories:",
      counterparty_summary["counterparty_country"].nunique())

display(counterparty_summary)

Number of unique counterparty categories: 252


,counterparty_country,series_count
0,All countries excluding residents,33
1,Netherlands,33
2,Singapore,33
3,Sweden,33
4,Portugal,33
...,...,...
247,Former Czechoslovakia,1
248,Former Soviet Union,1
249,Former Yugoslavia,1
250,Unallocated offshore centres,1


inspect reporting countries

In [31]:
reporting_summary = (
    filtered_bulk["Reporting country"]
    .value_counts(dropna=False)
    .rename_axis("reporting_country")
    .reset_index(name="series_count")
)

print("Number of unique reporting-country categories:",
      reporting_summary["reporting_country"].nunique())

display(reporting_summary)

Number of unique reporting-country categories: 33


,reporting_country,series_count
0,All reporting countries,252
1,Spain,232
2,Netherlands,231
3,France,229
4,India,229
5,United States,229
6,Canada,228
7,Italy,228
8,United Kingdom,228
9,Switzerland,225


# Step 3: identify aggregate and non-country categories

In [32]:
# Display all reporting-country categories
print("Reporting-country categories:")
display(
    pd.DataFrame({
        "reporting_country": sorted(
            filtered_bulk["Reporting country"].dropna().unique()
        )
    })
)

# Display counterparty categories containing likely aggregate/non-country terms
non_country_keywords = [
    "all ",
    "excluding",
    "resident",
    "residents",
    "international",
    "organisation",
    "organization",
    "unallocated",
    "local",
    "offshore",
    "world",
    "other"
]

counterparty_values = (
    filtered_bulk["Counterparty country"]
    .dropna()
    .astype(str)
    .unique()
)

possible_non_country = sorted([
    value for value in counterparty_values
    if any(keyword in value.lower() for keyword in non_country_keywords)
])

print("\nPossible aggregate or non-country counterparty categories:")
display(pd.DataFrame({
    "possible_non_country_category": possible_non_country
}))

Reporting-country categories:


,reporting_country
0,All reporting countries
1,Australia
2,Austria
3,Belgium
4,Brazil
5,Canada
6,Chile
7,Chinese Taipei
8,Denmark
9,Finland



Possible aggregate or non-country counterparty categories:


,possible_non_country_category
0,All countries
1,All countries excluding residents
2,International organisations
3,Marshall Islands
4,Residents/Local
5,Unallocated British Overseas Territories
6,Unallocated West Indies UK
7,Unallocated advanced economies
8,Unallocated emerging Africa and Middle East
9,Unallocated emerging Asia and Pacific


# Step 4: remove only clear aggregate categories

In [33]:
# Clear aggregate or non-country counterparty categories
excluded_counterparties = [
    "All countries",
    "All countries excluding residents",
    "International organisations",
    "Residents/Local",
    "Unallocated British Overseas Territories",
    "Unallocated West Indies UK",
    "Unallocated advanced economies",
    "Unallocated emerging Africa and Middle East",
    "Unallocated emerging Asia and Pacific",
    "Unallocated Latin America and Caribbean",
    "Unallocated offshore centres",
    "Unallocated other countries",
    "Unallocated world",
]

# Remove the aggregate reporting-country category
excluded_reporting_countries = [
    "All reporting countries"
]

network_candidates = filtered_bulk[
    ~filtered_bulk["Reporting country"].isin(excluded_reporting_countries)
    & ~filtered_bulk["Counterparty country"].isin(excluded_counterparties)
].copy()

print("Original filtered shape:", filtered_bulk.shape)
print("Network candidate shape:", network_candidates.shape)

print("\nReporting countries remaining:",
      network_candidates["Reporting country"].nunique())

print("Counterparty categories remaining:",
      network_candidates["Counterparty country"].nunique())

print("\nRemaining reporting countries:")
display(
    pd.DataFrame({
        "reporting_country": sorted(
            network_candidates["Reporting country"].unique()
        )
    })
)

print("\nRemaining counterparty categories containing 'Unallocated', 'All', or 'Residents':")
remaining_suspicious = network_candidates[
    network_candidates["Counterparty country"]
    .astype(str)
    .str.contains(
        "unallocated|all countries|residents|international",
        case=False,
        na=False
    )
]["Counterparty country"].unique()

display(pd.DataFrame({
    "remaining_category": sorted(remaining_suspicious)
}))

Original filtered shape: (6514, 167)
Network candidate shape: (6079, 167)

Reporting countries remaining: 32
Counterparty categories remaining: 228

Remaining reporting countries:


,reporting_country
0,Australia
1,Austria
2,Belgium
3,Brazil
4,Canada
5,Chile
6,Chinese Taipei
7,Denmark
8,Finland
9,France



Remaining counterparty categories containing 'Unallocated', 'All', or 'Residents':


,remaining_category
0,Unallocated location


# Step 5: remove the remaining residual category

In [34]:
# Remove the remaining residual/non-country counterparty category
network_candidates = network_candidates[
    network_candidates["Counterparty country"] != "Unallocated location"
].copy()

print("Network candidate shape after removing 'Unallocated location':",
      network_candidates.shape)

print("Reporting countries:",
      network_candidates["Reporting country"].nunique())

print("Counterparty categories:",
      network_candidates["Counterparty country"].nunique())

print("\nRemaining suspicious counterparty categories:")

suspicious_pattern = (
    "unallocated|all countries|residents|international|"
    "organisation|organization"
)

remaining_suspicious = sorted(
    network_candidates.loc[
        network_candidates["Counterparty country"]
        .astype(str)
        .str.contains(suspicious_pattern, case=False, na=False),
        "Counterparty country"
    ].unique()
)

display(pd.DataFrame({
    "remaining_suspicious_category": remaining_suspicious
}))

Network candidate shape after removing 'Unallocated location': (6048, 167)
Reporting countries: 32
Counterparty categories: 227

Remaining suspicious counterparty categories:


,remaining_suspicious_category


# Step 6: select the project period

In [35]:
analysis_periods = [
    "2007-Q1", "2007-Q2", "2007-Q3", "2007-Q4",
    "2008-Q1", "2008-Q2", "2008-Q3", "2008-Q4",
    "2009-Q1", "2009-Q2", "2009-Q3", "2009-Q4",
    "2010-Q1", "2010-Q2", "2010-Q3", "2010-Q4"
]

missing_periods = [
    period for period in analysis_periods
    if period not in network_candidates.columns
]

if missing_periods:
    print("Missing required periods:", missing_periods)
else:
    print("All required periods are available.")

network_wide = network_candidates[
    [
        "Series",
        "Reporting country",
        "Counterparty country",
        "Measure",
        "CBS reporting basis",
        "Balance sheet position",
        "CBS bank type",
        "Type of instruments",
        "Remaining maturity",
        "Currency type of booking location",
        "Counterparty sector",
    ] + analysis_periods
].copy()

print("\nNetwork-wide shape:", network_wide.shape)
display(network_wide.head())

All required periods are available.

Network-wide shape: (6048, 27)


,Series,Reporting country,Counterparty country,Measure,CBS reporting basis,Balance sheet position,CBS bank type,Type of instruments,Remaining maturity,Currency type of booking location,Counterparty sector,2007-Q1,2007-Q2,2007-Q3,2007-Q4,2008-Q1,2008-Q2,2008-Q3,2008-Q4,2009-Q1,2009-Q2,2009-Q3,2009-Q4,2010-Q1,2010-Q2,2010-Q3,2010-Q4
1217,Q:S:GR:4B:F:I:A:A:TO1:A:BI,Greece,Burundi,Amounts outstanding / Stocks,Immediate counterparty basis,International claims,Domestic banks,All instruments,Total (all maturities),All currencies,All sectors,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1218,Q:S:GR:4B:F:I:A:A:TO1:A:ET,Greece,Ethiopia,Amounts outstanding / Stocks,Immediate counterparty basis,International claims,Domestic banks,All instruments,Total (all maturities),All currencies,All sectors,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1219,Q:S:GR:4B:F:I:A:A:TO1:A:FJ,Greece,Fiji,Amounts outstanding / Stocks,Immediate counterparty basis,International claims,Domestic banks,All instruments,Total (all maturities),All currencies,All sectors,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1220,Q:S:GR:4B:F:I:A:A:TO1:A:GL,Greece,Greenland,Amounts outstanding / Stocks,Immediate counterparty basis,International claims,Domestic banks,All instruments,Total (all maturities),All currencies,All sectors,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1222,Q:S:GR:4B:F:I:A:A:TO1:A:IR,Greece,Iran,Amounts outstanding / Stocks,Immediate counterparty basis,International claims,Domestic banks,All instruments,Total (all maturities),All currencies,All sectors,NaN,1.0,1.0,2.0,1.0,1.0,2.0,2.0,2.0,2.0,5.0,4.0,4.0,4.0,4.0,4.0


calculate missingness across all 16 quarters

In [36]:
missing_summary = pd.DataFrame({
    "non_missing_count": network_wide[analysis_periods].notna().sum(),
    "missing_count": network_wide[analysis_periods].isna().sum(),
    "missing_percent": (
        network_wide[analysis_periods].isna().mean() * 100
    ).round(2),
    "zero_count": (network_wide[analysis_periods] == 0).sum(),
    "positive_count": (network_wide[analysis_periods] > 0).sum(),
})

display(missing_summary)

,non_missing_count,missing_count,missing_percent,zero_count,positive_count
2007-Q1,1741,4307,71.21,0,1741
2007-Q2,1755,4293,70.98,0,1752
2007-Q3,1776,4272,70.63,0,1773
2007-Q4,1793,4255,70.35,0,1791
2008-Q1,1785,4263,70.49,0,1785
2008-Q2,1788,4260,70.44,0,1788
2008-Q3,1796,4252,70.30,0,1796
2008-Q4,1774,4274,70.67,0,1774
2009-Q1,1789,4259,70.42,0,1789
2009-Q2,1790,4258,70.40,0,1790


Check the overall number of usable observations

In [37]:
total_cells = network_wide[analysis_periods].size
total_missing = network_wide[analysis_periods].isna().sum().sum()
total_non_missing = network_wide[analysis_periods].notna().sum().sum()
total_positive = (network_wide[analysis_periods] > 0).sum().sum()
total_zero = (network_wide[analysis_periods] == 0).sum().sum()

print(f"Total quarter cells: {total_cells:,}")
print(f"Missing cells: {total_missing:,}")
print(f"Non-missing cells: {total_non_missing:,}")
print(f"Positive cells: {total_positive:,}")
print(f"Zero cells: {total_zero:,}")
print(f"Missing percentage: {total_missing / total_cells * 100:.2f}%")

Total quarter cells: 96,768
Missing cells: 68,109
Non-missing cells: 28,659
Positive cells: 28,648
Zero cells: 0
Missing percentage: 70.38%


Inspect the rows with the most data

In [38]:
network_wide["non_missing_periods"] = network_wide[analysis_periods].notna().sum(axis=1)
network_wide["positive_periods"] = (network_wide[analysis_periods] > 0).sum(axis=1)

print("Rows with the most non-missing observations:")
display(
    network_wide.sort_values(
        ["non_missing_periods", "positive_periods"],
        ascending=False
    )[
        [
            "Reporting country",
            "Counterparty country",
            "non_missing_periods",
            "positive_periods",
        ] + analysis_periods
    ].head(10)
)

Rows with the most non-missing observations:


,Reporting country,Counterparty country,non_missing_periods,positive_periods,2007-Q1,2007-Q2,2007-Q3,2007-Q4,2008-Q1,2008-Q2,2008-Q3,2008-Q4,2009-Q1,2009-Q2,2009-Q3,2009-Q4,2010-Q1,2010-Q2,2010-Q3,2010-Q4
1239,Greece,"St Helena, Ascension and Tristan da Cunha",16,16,1.0,1.0,1.0,1.0,2.0,2.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
1249,Greece,Malta,16,16,183.0,319.0,298.0,589.0,528.0,658.0,674.0,567.0,533.0,435.0,390.0,434.0,448.0,459.0,465.0,466.0
1253,Greece,Malaysia,16,16,1.0,1.0,1.0,2.0,3.0,2.0,3.0,3.0,3.0,3.0,1.0,1.0,1.0,1.0,1.0,1.0
1257,Greece,Netherlands,16,16,1888.0,2499.0,1011.0,817.0,1040.0,815.0,1264.0,1224.0,1023.0,1212.0,1183.0,5439.0,5101.0,4231.0,4498.0,4244.0
1258,Greece,Norway,16,16,487.0,291.0,365.0,20.0,47.0,35.0,88.0,93.0,76.0,84.0,90.0,184.0,92.0,86.0,86.0,80.0
1263,Greece,Panama,16,16,343.0,683.0,501.0,612.0,879.0,848.0,734.0,1062.0,852.0,760.0,803.0,1053.0,1009.0,1094.0,1128.0,1439.0
1272,Greece,Poland,16,16,34.0,40.0,35.0,37.0,40.0,39.0,37.0,35.0,35.0,49.0,41.0,4944.0,4624.0,4873.0,5304.0,5399.0
1274,Greece,Portugal,16,16,10.0,21.0,19.0,19.0,67.0,19.0,27.0,9.0,29.0,53.0,308.0,105.0,91.0,101.0,150.0,82.0
1277,Greece,Qatar,16,16,31.0,33.0,34.0,36.0,41.0,44.0,48.0,50.0,50.0,50.0,50.0,50.0,49.0,49.0,48.0,48.0
1279,Greece,Romania,16,16,5434.0,6695.0,9821.0,11913.0,14136.0,16323.0,14635.0,16716.0,15414.0,15459.0,15214.0,18929.0,17616.0,16002.0,17093.0,16141.0


inspect values and units

In [39]:
# Inspect the distribution of observed values across the 16 project quarters

quarter_values = network_wide[analysis_periods].stack().dropna()

print("Number of observed values:", len(quarter_values))
print("Minimum value:", quarter_values.min())
print("Maximum value:", quarter_values.max())
print("Median value:", quarter_values.median())

print("\nSelected percentiles:")
display(
    quarter_values.quantile(
        [0, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99, 1.00]
    ).rename("value").to_frame()
)

Number of observed values: 28659
Minimum value: -440.0
Maximum value: 1588942.0
Median value: 251.0

Selected percentiles:


,value
0.00,-440.00
0.25,22.00
0.50,251.00
0.75,2722.50
0.90,16468.00
0.95,41445.70
0.99,214635.92
1.00,1588942.00


In [40]:
# Find the largest observed country-pair exposures in the project period

id_columns = [
    "Reporting country",
    "Counterparty country"
]

top_observations = (
    network_wide[
        id_columns + analysis_periods
    ]
    .melt(
        id_vars=id_columns,
        value_vars=analysis_periods,
        var_name="period",
        value_name="claim_value"
    )
    .dropna(subset=["claim_value"])
    .sort_values("claim_value", ascending=False)
)

display(top_observations.head(20))

,Reporting country,Counterparty country,period,claim_value
32935,Germany,Euro area,2008-Q2,1588942.0
26887,Germany,Euro area,2008-Q1,1579108.0
38983,Germany,Euro area,2008-Q3,1449904.0
20839,Germany,Euro area,2007-Q4,1449596.0
14791,Germany,Euro area,2007-Q3,1354708.0
63175,Germany,Euro area,2009-Q3,1309688.0
8743,Germany,Euro area,2007-Q2,1305063.0
45031,Germany,Euro area,2008-Q4,1288872.0
57127,Germany,Euro area,2009-Q2,1253439.0
69223,Germany,Euro area,2009-Q4,1240261.0


inspect all counterparty categories containing regional terms

In [41]:
regional_keywords = [
    "euro",
    "area",
    "region",
    "advanced economies",
    "emerging",
    "africa",
    "asia",
    "pacific",
    "latin america",
    "caribbean",
    "middle east",
    "offshore",
    "world",
    "other"
]

all_counterparties = sorted(
    network_wide["Counterparty country"]
    .dropna()
    .astype(str)
    .unique()
)

possible_regional_aggregates = [
    value for value in all_counterparties
    if any(keyword in value.lower() for keyword in regional_keywords)
]

print("Possible regional or aggregate counterparty categories:")
display(pd.DataFrame({
    "category": possible_regional_aggregates
}))

Possible regional or aggregate counterparty categories:


,category
0,Central African Republic
1,Euro area
2,South Africa
3,U.S. Miscellaneous Pacific Islands


inspect large exposures by category

In [42]:
largest_by_counterparty = (
    top_observations
    .groupby("Counterparty country", as_index=False)["claim_value"]
    .max()
    .sort_values("claim_value", ascending=False)
)

display(largest_by_counterparty.head(30))

,Counterparty country,claim_value
62,Euro area,1588942.0
203,United States,768799.0
202,United Kingdom,712338.0
33,Cayman Islands,319825.0
174,Spain,273201.0
67,France,246974.0
91,Ireland,231077.0
94,Italy,220336.0
71,Germany,203255.0
135,Netherlands,141928.0


# Step 7: remove `Euro area`

In [43]:
# Remove the regional aggregate Euro area
network_country_wide = network_wide[
    network_wide["Counterparty country"] != "Euro area"
].copy()

print("Shape after removing Euro area:",
      network_country_wide.shape)

print("Reporting countries:",
      network_country_wide["Reporting country"].nunique())

print("Counterparty categories:",
      network_country_wide["Counterparty country"].nunique())

print("\nRemaining regional/aggregate candidates:")

remaining_candidates = sorted(
    network_country_wide.loc[
        network_country_wide["Counterparty country"].astype(str).str.contains(
            "euro area|all countries|unallocated|residents/local|"
            "international organisations",
            case=False,
            na=False
        ),
        "Counterparty country"
    ].unique()
)

display(pd.DataFrame({
    "remaining_category": remaining_candidates
}))

Shape after removing Euro area: (6016, 29)
Reporting countries: 32
Counterparty categories: 226

Remaining regional/aggregate candidates:


,remaining_category


# Step 8: remove temporary inspection columns

In [44]:
temporary_columns = [
    "non_missing_periods",
    "positive_periods"
]

network_country_wide = network_country_wide.drop(
    columns=[
        column for column in temporary_columns
        if column in network_country_wide.columns
    ]
)

print("Clean wide dataset shape:", network_country_wide.shape)
print("Columns:", len(network_country_wide.columns))

display(network_country_wide.head())

Clean wide dataset shape: (6016, 27)
Columns: 27


,Series,Reporting country,Counterparty country,Measure,CBS reporting basis,Balance sheet position,CBS bank type,Type of instruments,Remaining maturity,Currency type of booking location,Counterparty sector,2007-Q1,2007-Q2,2007-Q3,2007-Q4,2008-Q1,2008-Q2,2008-Q3,2008-Q4,2009-Q1,2009-Q2,2009-Q3,2009-Q4,2010-Q1,2010-Q2,2010-Q3,2010-Q4
1217,Q:S:GR:4B:F:I:A:A:TO1:A:BI,Greece,Burundi,Amounts outstanding / Stocks,Immediate counterparty basis,International claims,Domestic banks,All instruments,Total (all maturities),All currencies,All sectors,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1218,Q:S:GR:4B:F:I:A:A:TO1:A:ET,Greece,Ethiopia,Amounts outstanding / Stocks,Immediate counterparty basis,International claims,Domestic banks,All instruments,Total (all maturities),All currencies,All sectors,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1219,Q:S:GR:4B:F:I:A:A:TO1:A:FJ,Greece,Fiji,Amounts outstanding / Stocks,Immediate counterparty basis,International claims,Domestic banks,All instruments,Total (all maturities),All currencies,All sectors,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1220,Q:S:GR:4B:F:I:A:A:TO1:A:GL,Greece,Greenland,Amounts outstanding / Stocks,Immediate counterparty basis,International claims,Domestic banks,All instruments,Total (all maturities),All currencies,All sectors,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1222,Q:S:GR:4B:F:I:A:A:TO1:A:IR,Greece,Iran,Amounts outstanding / Stocks,Immediate counterparty basis,International claims,Domestic banks,All instruments,Total (all maturities),All currencies,All sectors,NaN,1.0,1.0,2.0,1.0,1.0,2.0,2.0,2.0,2.0,5.0,4.0,4.0,4.0,4.0,4.0


# Step 9: verify the final wide structure

In [45]:
expected_metadata_columns = [
    "Series",
    "Reporting country",
    "Counterparty country",
    "Measure",
    "CBS reporting basis",
    "Balance sheet position",
    "CBS bank type",
    "Type of instruments",
    "Remaining maturity",
    "Currency type of booking location",
    "Counterparty sector",
]

missing_metadata_columns = [
    column for column in expected_metadata_columns
    if column not in network_country_wide.columns
]

missing_analysis_periods = [
    period for period in analysis_periods
    if period not in network_country_wide.columns
]

print("Missing metadata columns:", missing_metadata_columns)
print("Missing analysis periods:", missing_analysis_periods)

print("\nFinal wide dataset columns:")
print(network_country_wide.columns.tolist())

Missing metadata columns: []
Missing analysis periods: []

Final wide dataset columns:
['Series', 'Reporting country', 'Counterparty country', 'Measure', 'CBS reporting basis', 'Balance sheet position', 'CBS bank type', 'Type of instruments', 'Remaining maturity', 'Currency type of booking location', 'Counterparty sector', '2007-Q1', '2007-Q2', '2007-Q3', '2007-Q4', '2008-Q1', '2008-Q2', '2008-Q3', '2008-Q4', '2009-Q1', '2009-Q2', '2009-Q3', '2009-Q4', '2010-Q1', '2010-Q2', '2010-Q3', '2010-Q4']


# Step 10: reshape wide data to long format

In [46]:
metadata_columns = [
    "Series",
    "Reporting country",
    "Counterparty country",
    "Measure",
    "CBS reporting basis",
    "Balance sheet position",
    "CBS bank type",
    "Type of instruments",
    "Remaining maturity",
    "Currency type of booking location",
    "Counterparty sector",
]

network_long = network_country_wide.melt(
    id_vars=metadata_columns,
    value_vars=analysis_periods,
    var_name="period",
    value_name="claim_value_usd_millions"
)

print("Long dataset shape before removing missing values:",
      network_long.shape)

display(network_long.head(10))

Long dataset shape before removing missing values: (96256, 13)


,Series,Reporting country,Counterparty country,Measure,CBS reporting basis,Balance sheet position,CBS bank type,Type of instruments,Remaining maturity,Currency type of booking location,Counterparty sector,period,claim_value_usd_millions
0,Q:S:GR:4B:F:I:A:A:TO1:A:BI,Greece,Burundi,Amounts outstanding / Stocks,Immediate counterparty basis,International claims,Domestic banks,All instruments,Total (all maturities),All currencies,All sectors,2007-Q1,NaN
1,Q:S:GR:4B:F:I:A:A:TO1:A:ET,Greece,Ethiopia,Amounts outstanding / Stocks,Immediate counterparty basis,International claims,Domestic banks,All instruments,Total (all maturities),All currencies,All sectors,2007-Q1,NaN
2,Q:S:GR:4B:F:I:A:A:TO1:A:FJ,Greece,Fiji,Amounts outstanding / Stocks,Immediate counterparty basis,International claims,Domestic banks,All instruments,Total (all maturities),All currencies,All sectors,2007-Q1,NaN
3,Q:S:GR:4B:F:I:A:A:TO1:A:GL,Greece,Greenland,Amounts outstanding / Stocks,Immediate counterparty basis,International claims,Domestic banks,All instruments,Total (all maturities),All currencies,All sectors,2007-Q1,NaN
4,Q:S:GR:4B:F:I:A:A:TO1:A:IR,Greece,Iran,Amounts outstanding / Stocks,Immediate counterparty basis,International claims,Domestic banks,All instruments,Total (all maturities),All currencies,All sectors,2007-Q1,NaN
5,Q:S:GR:4B:F:I:A:A:TO1:A:KN,Greece,St Kitts and Nevis,Amounts outstanding / Stocks,Immediate counterparty basis,International claims,Domestic banks,All instruments,Total (all maturities),All currencies,All sectors,2007-Q1,NaN
6,Q:S:GR:4B:F:I:A:A:TO1:A:KP,Greece,North Korea,Amounts outstanding / Stocks,Immediate counterparty basis,International claims,Domestic banks,All instruments,Total (all maturities),All currencies,All sectors,2007-Q1,NaN
7,Q:S:GR:4B:F:I:A:A:TO1:A:LS,Greece,Lesotho,Amounts outstanding / Stocks,Immediate counterparty basis,International claims,Domestic banks,All instruments,Total (all maturities),All currencies,All sectors,2007-Q1,NaN
8,Q:S:GR:4B:F:I:A:A:TO1:A:MR,Greece,Mauritania,Amounts outstanding / Stocks,Immediate counterparty basis,International claims,Domestic banks,All instruments,Total (all maturities),All currencies,All sectors,2007-Q1,NaN
9,Q:S:GR:4B:F:I:A:A:TO1:A:MW,Greece,Malawi,Amounts outstanding / Stocks,Immediate counterparty basis,International claims,Domestic banks,All instruments,Total (all maturities),All currencies,All sectors,2007-Q1,NaN


# Step 11: inspect missing values in the long dataset

In [47]:
long_missing_summary = pd.DataFrame({
    "count": network_long.isna().sum(),
    "percent": (network_long.isna().mean() * 100).round(2),
    "dtype": network_long.dtypes.astype(str)
}).sort_values("count", ascending=False)

display(long_missing_summary)

,count,percent,dtype
claim_value_usd_millions,67980,70.62,float64
Reporting country,0,0.00,object
Counterparty country,0,0.00,object
Measure,0,0.00,object
Series,0,0.00,object
CBS reporting basis,0,0.00,object
Balance sheet position,0,0.00,object
Type of instruments,0,0.00,object
CBS bank type,0,0.00,object
Remaining maturity,0,0.00,object


# Step 12: count observed and missing claims

In [48]:
claim_missing_count = network_long["claim_value_usd_millions"].isna().sum()
claim_observed_count = network_long["claim_value_usd_millions"].notna().sum()

print("Total long-format rows:", len(network_long))
print("Observed claim values:", claim_observed_count)
print("Missing claim values:", claim_missing_count)
print(
    "Missing claim percentage:",
    round(claim_missing_count / len(network_long) * 100, 2),
    "%"
)

Total long-format rows: 96256
Observed claim values: 28276
Missing claim values: 67980
Missing claim percentage: 70.62 %


# Step 13: inspect negative values

In [49]:
negative_claims = network_long[
    network_long["claim_value_usd_millions"] < 0
].copy()

print("Number of negative observations:", len(negative_claims))

display(
    negative_claims[
        [
            "Reporting country",
            "Counterparty country",
            "period",
            "claim_value_usd_millions"
        ]
    ]
    .sort_values("claim_value_usd_millions")
    .head(20)
)

Number of negative observations: 11


,Reporting country,Counterparty country,period,claim_value_usd_millions
7924,Denmark,Lithuania,2007-Q2,-440.0
74072,Denmark,Austria,2010-Q1,-236.0
13940,Denmark,Lithuania,2007-Q3,-155.0
68056,Denmark,Austria,2009-Q4,-61.0
12948,United Kingdom,Papua New Guinea,2007-Q3,-47.0
6932,United Kingdom,Papua New Guinea,2007-Q2,-38.0
18964,United Kingdom,Papua New Guinea,2007-Q4,-26.0
18906,United Kingdom,Fiji,2007-Q4,-4.0
6874,United Kingdom,Fiji,2007-Q2,-2.0
12890,United Kingdom,Fiji,2007-Q3,-2.0


# Step 14: check duplicate country-pair-period observations

In [50]:
duplicate_keys = [
    "Reporting country",
    "Counterparty country",
    "period"
]

duplicate_rows = network_long[
    network_long.duplicated(subset=duplicate_keys, keep=False)
].copy()

print("Rows involved in duplicate country-pair-period keys:",
      len(duplicate_rows))

if len(duplicate_rows) > 0:
    display(
        duplicate_rows[
            duplicate_keys + ["claim_value_usd_millions", "Series"]
        ].sort_values(duplicate_keys).head(30)
    )
else:
    print("No duplicate country-pair-period observations found.")

Rows involved in duplicate country-pair-period keys: 0
No duplicate country-pair-period observations found.


# Step 15: create analysis datasets

1. `network_long_all`: all rows, including missing and negative observations.

2. `network_observed`: only rows with a reported numerical value, including negative values.

3. `network_positive`: only rows with strictly positive claims, suitable for the lending network.

In [51]:
network_long_all = network_long.copy()

network_observed = network_long[
    network_long["claim_value_usd_millions"].notna()
].copy()

network_positive = network_observed[
    network_observed["claim_value_usd_millions"] > 0
].copy()

network_negative = network_observed[
    network_observed["claim_value_usd_millions"] < 0
].copy()

print("All long-format rows:", len(network_long_all))
print("Observed rows:", len(network_observed))
print("Positive rows:", len(network_positive))
print("Negative rows:", len(network_negative))
print("Zero rows:", (network_observed["claim_value_usd_millions"] == 0).sum())

All long-format rows: 96256
Observed rows: 28276
Positive rows: 28265
Negative rows: 11
Zero rows: 0


# Step 16: calculate missingness and positive-edge coverage by quarter

In [52]:
quarter_quality = (
    network_long
    .groupby("period")
    .agg(
        total_series=("claim_value_usd_millions", "size"),
        observed_values=("claim_value_usd_millions", "count"),
        positive_values=(
            "claim_value_usd_millions",
            lambda x: (x > 0).sum()
        ),
        negative_values=(
            "claim_value_usd_millions",
            lambda x: (x < 0).sum()
        )
    )
    .reset_index()
)

quarter_quality["missing_values"] = (
    quarter_quality["total_series"]
    - quarter_quality["observed_values"]
)

quarter_quality["missing_percent"] = (
    quarter_quality["missing_values"]
    / quarter_quality["total_series"]
    * 100
).round(2)

display(quarter_quality)

,period,total_series,observed_values,positive_values,negative_values,missing_values,missing_percent
0,2007-Q1,6016,1719,1719,0,4297,71.43
1,2007-Q2,6016,1733,1730,3,4283,71.19
2,2007-Q3,6016,1752,1749,3,4264,70.88
3,2007-Q4,6016,1769,1767,2,4247,70.60
4,2008-Q1,6016,1761,1761,0,4255,70.73
5,2008-Q2,6016,1764,1764,0,4252,70.68
6,2008-Q3,6016,1772,1772,0,4244,70.55
7,2008-Q4,6016,1750,1750,0,4266,70.91
8,2009-Q1,6016,1765,1765,0,4251,70.66
9,2009-Q2,6016,1766,1766,0,4250,70.64


## Interpreting Missing Values and Network Coverage

The reshaped dataset contains 6,016 possible reporting-country and counterparty-country series observed across 16 quarters. This produces 96,256 possible country-pair-quarter cells.

Of these cells:

- 28,276 contain reported numerical values.
- 28,265 contain positive claims.
- 11 contain negative values.
- 67,980 are missing.
- No observed values are equal to zero.
- No duplicate reporting-country, counterparty-country, and period combinations exist.

The overall missing-cell percentage is 70.62%. This is high when calculated across the complete rectangular table, but it does not mean that 70% of the banking system is missing. It means that many possible country-pair-quarter combinations have no reported observation. A reporting banking system may have international claims to a limited set of counterparties while having no reported series for many other possible counterparties.

The missingness is also stable across the project period. Each quarter contains approximately 1,700–1,800 observed positive relationships, and the missing percentage remains close to 70% in every quarter. There is therefore no evidence of a sudden data-coverage collapse during the 2008–09 crisis period.

### Treatment of missing observations

Missing observations are retained in the audit version of the dataset and are not automatically converted to zero. A missing value may represent an unavailable or unreported series, a relationship below a reporting threshold, confidentiality treatment, or a series that does not apply to a particular country-period combination. Because the dataset contains no explicit zero values, the analysis does not assume that missing values represent zero lending.

For network construction, only positive reported claims are treated as lending edges:

\[
\text{Reporting country} \rightarrow \text{Counterparty country}
\]

The absence of a reported edge is not interpreted as proof that no relationship existed. This distinction is important for hub identification, resilience analysis, and causal interpretation.

### Treatment of negative observations

Only 11 negative observations are present, compared with 28,276 observed values. These values are retained in the observed-data audit because they may reflect adjustments, revisions, breaks, or other reporting effects. They are excluded from the positive-edge network because a lending edge in this project represents a positive reported claim.

### Analytical datasets

Three versions are maintained:

1. `network_long_all` contains every country-pair-quarter combination, including missing values.
2. `network_observed` contains all reported numerical observations, including the 11 negative values.
3. `network_positive` contains only positive reported claims and is the primary dataset for network analysis.

This approach preserves transparency while ensuring that network measures are calculated from actual positive reported exposures rather than from unverified assumptions about missing relationships.

# Step 17: create the final positive-edge dataset

In [53]:
network_positive = network_long[
    network_long["claim_value_usd_millions"] > 0
].copy()

network_positive = network_positive.sort_values(
    ["period", "Reporting country", "Counterparty country"]
).reset_index(drop=True)

print("Positive-edge dataset shape:", network_positive.shape)

print("\nPositive observations by period:")
display(
    network_positive["period"]
    .value_counts()
    .reindex(analysis_periods)
    .rename("positive_edge_count")
    .to_frame()
)

Positive-edge dataset shape: (28265, 13)

Positive observations by period:


,positive_edge_count
period,
2007-Q1,1719
2007-Q2,1730
2007-Q3,1749
2007-Q4,1767
2008-Q1,1761
2008-Q2,1764
2008-Q3,1772
2008-Q4,1750
2009-Q1,1765


# Step 18: check the final positive-edge data

In [54]:
print("Missing claim values:",
      network_positive["claim_value_usd_millions"].isna().sum())

print("Non-positive claim values:",
      (network_positive["claim_value_usd_millions"] <= 0).sum())

print("Duplicate country-pair-period keys:",
      network_positive.duplicated(
          subset=[
              "Reporting country",
              "Counterparty country",
              "period"
          ]
      ).sum())

display(network_positive.head(10))

Missing claim values: 0
Non-positive claim values: 0
Duplicate country-pair-period keys: 0


,Series,Reporting country,Counterparty country,Measure,CBS reporting basis,Balance sheet position,CBS bank type,Type of instruments,Remaining maturity,Currency type of booking location,Counterparty sector,period,claim_value_usd_millions
0,Q:S:AU:4B:F:I:A:A:TO1:A:AR,Australia,Argentina,Amounts outstanding / Stocks,Immediate counterparty basis,International claims,Domestic banks,All instruments,Total (all maturities),All currencies,All sectors,2007-Q1,2.0
1,Q:S:AU:4B:F:I:A:A:TO1:A:AT,Australia,Austria,Amounts outstanding / Stocks,Immediate counterparty basis,International claims,Domestic banks,All instruments,Total (all maturities),All currencies,All sectors,2007-Q1,314.0
2,Q:S:AU:4B:F:I:A:A:TO1:A:BH,Australia,Bahrain,Amounts outstanding / Stocks,Immediate counterparty basis,International claims,Domestic banks,All instruments,Total (all maturities),All currencies,All sectors,2007-Q1,18.0
3,Q:S:AU:4B:F:I:A:A:TO1:A:BD,Australia,Bangladesh,Amounts outstanding / Stocks,Immediate counterparty basis,International claims,Domestic banks,All instruments,Total (all maturities),All currencies,All sectors,2007-Q1,132.0
4,Q:S:AU:4B:F:I:A:A:TO1:A:BE,Australia,Belgium,Amounts outstanding / Stocks,Immediate counterparty basis,International claims,Domestic banks,All instruments,Total (all maturities),All currencies,All sectors,2007-Q1,275.0
5,Q:S:AU:4B:F:I:A:A:TO1:A:BM,Australia,Bermuda,Amounts outstanding / Stocks,Immediate counterparty basis,International claims,Domestic banks,All instruments,Total (all maturities),All currencies,All sectors,2007-Q1,998.0
6,Q:S:AU:4B:F:I:A:A:TO1:A:BR,Australia,Brazil,Amounts outstanding / Stocks,Immediate counterparty basis,International claims,Domestic banks,All instruments,Total (all maturities),All currencies,All sectors,2007-Q1,238.0
7,Q:S:AU:4B:F:I:A:A:TO1:A:BN,Australia,Brunei,Amounts outstanding / Stocks,Immediate counterparty basis,International claims,Domestic banks,All instruments,Total (all maturities),All currencies,All sectors,2007-Q1,13.0
8,Q:S:AU:4B:F:I:A:A:TO1:A:KH,Australia,Cambodia,Amounts outstanding / Stocks,Immediate counterparty basis,International claims,Domestic banks,All instruments,Total (all maturities),All currencies,All sectors,2007-Q1,2.0
9,Q:S:AU:4B:F:I:A:A:TO1:A:CA,Australia,Canada,Amounts outstanding / Stocks,Immediate counterparty basis,International claims,Domestic banks,All instruments,Total (all maturities),All currencies,All sectors,2007-Q1,1049.0


# Step 19: Save the three prepared datasets

In [ ]:
from pathlib import Path

PROCESSED_DIR = Path("../data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("Processed-data folder:", PROCESSED_DIR.resolve())
print("Folder exists:", PROCESSED_DIR.exists())

In [56]:
network_long_all.to_csv(
    PROCESSED_DIR / "bis_cbs_network_2007_2010_all.csv",
    index=False
)

network_observed.to_csv(
    PROCESSED_DIR / "bis_cbs_network_2007_2010_observed.csv",
    index=False
)

network_positive.to_csv(
    PROCESSED_DIR / "bis_cbs_network_2007_2010_positive_edges.csv",
    index=False
)

quarter_quality.to_csv(
    PROCESSED_DIR / "bis_cbs_network_2007_2010_data_quality.csv",
    index=False
)

print("Saved files:")
for file_path in sorted(PROCESSED_DIR.glob("*.csv")):
    size_kb = file_path.stat().st_size / 1024
    print(f"- {file_path.name}: {size_kb:,.1f} KB")

Saved files:
- bis_cbs_network_2007_2010_all.csv: 20,451.4 KB
- bis_cbs_network_2007_2010_data_quality.csv: 0.7 KB
- bis_cbs_network_2007_2010_observed.csv: 6,084.5 KB
- bis_cbs_network_2007_2010_positive_edges.csv: 6,082.1 KB


## Reference:
Bank for International Settlements. (2026). Consolidated banking statistics: Detailed view of positions on individual countries (Table B4) [Data set]. BIS Data Portal. Retrieved August 25, 2026, from https://data.bis.org/topics/CBS/data

Bank for International Settlements. (2026). Consolidated banking statistics [Data set]. BIS Data Portal. Retrieved August 25, 2026, from https://www.bis.org/statistics/consstats.htm

Bank for International Settlements. (2026). Consolidated banking statistics bulk download [Data set]. BIS Data Portal. Retrieved August 25, 2026, from https://data.bis.org/bulkdownload